# Classification without fine-tuning: the code

Short versions of each method: one request to each API, and one training run each for TF-IDF and
BERTić. The prompts, inputs and model versions are the same as in the full experiment. Retries,
resume and result storage are left out. The full runs are in `run_jev.py`, `run_openai.py`,
`run_llm.py`, `tfidf.py` and `bertic_train.py`.

You need the API keys in `.env` and an NVIDIA GPU for BERTić. Start with `just notebook`.

In [1]:
import json
import os
import time

import httpx
import pandas as pd

from common import load, load_categories, load_env

pd.set_option("display.max_colwidth", None)
load_env()  # API keys from .env
df = load()  # products with their labels and the train/dev/test split
categories = load_categories()
print(f"{len(df):,} products, {len(categories)} categories")

29,684 products, 10 categories


## The data

The 10 categories. For the zero-shot methods, the description is the only definition of a
category.

In [2]:
count = categories.id.map(df.category.value_counts())
categories.assign(products=count, share=(count / len(df)).map("{:.1%}".format))

,id,name_hr,description,products,share
0,food,Hrana,"Fruit and vegetables, meat, fish, dairy, eggs, bread and bakery, frozen and canned food, pasta, rice, flour, oil, spices, sauces, sweets and snacks; excludes baby food and supplements",14204,47.9%
1,drinks,"Bezalkoholna pića, kava i čaj","Water, juices, soft drinks, energy drinks, iced tea, syrups, instant drink powders; coffee, tea and cocoa in any form",2563,8.6%
2,alcohol,Alkoholna pića,"Beer, cider, wine, sparkling wine, spirits, liqueurs",3082,10.4%
3,cosmetics,Kozmetika,"Hair, skin, body and oral care, deodorants, shaving, make-up, perfume",4443,15.0%
4,toiletries,Toaletne potrepštine,"Toilet paper, tissues, wet wipes, pads, tampons, cotton wool and buds",540,1.8%
5,cleaning,Sredstva za čišćenje,"Laundry and dish detergents, softeners, surface, WC and glass cleaners, air fresheners",2053,6.9%
6,household,Proizvodi za kućanstvo,"Kitchenware, foil and bags, cleaning tools, bulbs and batteries, candles, stationery, textile, garden, DIY, seasonal decorations",1269,4.3%
7,baby,Bebe i djeca,"Baby food and formula, diapers, baby cosmetics and care, dummies and bottles, toys",739,2.5%
8,health,Zdravlje,"Supplements and vitamins, OTC remedies, plasters and first aid, condoms, contact lenses",142,0.5%
9,pets,Kućni ljubimci,"Pet food, litter, pet accessories",649,2.2%


Random products. Every method gets the same input: the product name and the pack size. There is
no separate brand field, but many names contain the brand.

In [3]:
df.sample(10, random_state=7)[["name", "quantity", "unit", "category"]].reset_index(drop=True)

,name,quantity,unit,category
0,L'Oréal Casting Creme Gloss Boja za kosu 200 Ebanovina,1,kom,cosmetics
1,Paprika Fileti Žuta,0.530,kg,food
2,Voćno vino kupine,0.75,L,alcohol
3,"LCA Probiotik Jogurt Borovnica 1,2% m.m.",180,g,food
4,Teranino liker od vina Teran,0.5,L,alcohol
5,Deo Stick Dove Men Clean Comfort,50.0,ml,cosmetics
6,Grickalica Set 2/1,1,kom,cosmetics
7,Lenor Omekšivač Sensitive,0.77,L,cleaning
8,Ulje za kosu Argan,0.118,L,cosmetics
9,Panettone Classic,0.900,kg,food


## Demo products

Five products from the test split, so the trained models do not see them in training:

- **Pelinkovac**: a herbal liqueur. The easy case.
- **Kraš Express Kakao Napitak**: instant cocoa. The `drinks` description says "cocoa in any form".
- **Batida Maracuja**: a passion fruit liqueur. The name does not say that it contains alcohol.
- **Tomi Losos/Pastrva**: pet food. The name says only "salmon/trout"; Tomi is a pet food brand.
- **Violeta XXL Ručnici**: paper towels. The labels put paper towels in `toiletries`, but the
  `toiletries` description does not list them (EXPERIMENT.md, section 3, "Labels and the
  specification").

`classify()` runs one method on these products and prints the category, the confidence and the
time per product.

In [4]:
DEMO = ["3850158406356", "3850102126903", "4003310017177", "4003024157053", "3859892288129"]
demo = df.set_index("ean").loc[DEMO, ["name", "quantity", "unit", "category"]]
products = demo.to_dict("records")
live = {}  # the answers of each method, compared at the end


def classify(ask) -> list[str]:
    """Run ask(product) -> (category, confidence) on the demo products."""
    answers = []
    for p in products:
        t0 = time.perf_counter()
        category, confidence = ask(p)
        ms = (time.perf_counter() - t0) * 1000
        mark = "✓" if category == p["category"] else f"✗ (label: {p['category']})"
        print(f"{p['name']:28s} {category:10s} {confidence:4.2f} {ms:8.1f} ms  {mark}")
        answers.append(category)
    return answers


demo

,name,quantity,unit,category
ean,,,,
3850158406356,Pelinkovac,0.4,L,alcohol
3850102126903,Kraš Express Kakao Napitak,200,g,drinks
4003310017177,Batida Maracuja,0.25,L,alcohol
4003024157053,Tomi Losos/Pastrva,400,g,pets
3859892288129,Violeta XXL Ručnici,1,kom,toiletries


## The prompt

All API methods get the same instruction (prompt `v2` in `config/`) and the category descriptions
from the table above.

In [5]:
INSTRUCTIONS = (
    "The product is from a Croatian retail store. The name is in Croatian, as the store lists it, "
    "and can contain the brand. Which store category does this product belong to?"
)
DESCRIPTIONS = dict(zip(categories.id, categories.description))

## Jev (TypeSafe)

Jev gets the product as structured data (the state) and a question with the categories as
criteria. It answers with one category and a probability for each category.

In [6]:
from typesafe_sdk import Choice, TypeSafeClient

jev = TypeSafeClient(model="jev-1.14.0")  # API key from TYPESAFE_API_KEY
question = Choice(instructions=INSTRUCTIONS, criteria=DESCRIPTIONS)


def jev_answer(p):
    resp = jev.system_one(
        state={"product": p["name"], "quantity": f"{p['quantity']} {p['unit']}"},
        questions={"category": question},
    )
    return resp.choices["category"]


jev_answer(products[0])

ChoiceAnswer(type='choice', choice='alcohol', confidence=1.0, probabilities={'food': 0.0, 'drinks': 0.0, 'alcohol': 1.0, 'cosmetics': 0.0, 'toiletries': 0.0, 'cleaning': 0.0, 'household': 0.0, 'baby': 0.0, 'health': 0.0, 'pets': 0.0})

In [7]:
def ask_jev(p):
    a = jev_answer(p)
    return a.choice, a.confidence


live["jev"] = classify(ask_jev)

Pelinkovac                   alcohol    1.00    172.9 ms  ✓
Kraš Express Kakao Napitak   drinks     1.00    179.1 ms  ✓
Batida Maracuja              alcohol    0.96    179.5 ms  ✓
Tomi Losos/Pastrva           food       0.99    192.6 ms  ✗ (label: pets)
Violeta XXL Ručnici          household  0.96    157.8 ms  ✗ (label: toiletries)


## GPT-6 Luna (OpenAI Decisions API)

The same idea as Jev: a question with a list of choices, and a probability for each choice. The
input is the product as text. This is a plain HTTP request to `POST /v1/decisions`.

In [8]:
openai_api = httpx.Client(base_url="https://api.openai.com/v1", timeout=60,
                          headers={"Authorization": f"Bearer {os.environ['OPENAI_API_KEY']}"})
luna_question = {
    "name": "category",
    "type": "choice",
    "instructions": INSTRUCTIONS,
    "choices": [{"value": c, "description": d} for c, d in DESCRIPTIONS.items()],
}


def luna_answer(p):
    r = openai_api.post("/decisions", json={
        "model": "gpt-6-luna",
        "input": f"Product: {p['name']}\nQuantity: {p['quantity']} {p['unit']}",
        "questions": [luna_question],
    })
    r.raise_for_status()
    return r.json()["answers"][0]


luna_answer(products[0])

{'type': 'choice',
 'name': 'category',
 'choice': 'alcohol',
 'probabilities': [{'value': 'food', 'probability': 0.01},
  {'value': 'drinks', 'probability': 0.01},
  {'value': 'alcohol', 'probability': 0.98},
  {'value': 'cosmetics', 'probability': 0.0},
  {'value': 'toiletries', 'probability': 0.0},
  {'value': 'cleaning', 'probability': 0.0},
  {'value': 'household', 'probability': 0.0},
  {'value': 'baby', 'probability': 0.0},
  {'value': 'health', 'probability': 0.0},
  {'value': 'pets', 'probability': 0.0}],
 'confidence': 0.98}

In [9]:
def ask_luna(p):
    a = luna_answer(p)
    return a["choice"], a["confidence"]


live["luna"] = classify(ask_luna)

Pelinkovac                   alcohol    0.98    140.7 ms  ✓
Kraš Express Kakao Napitak   drinks     0.94    248.8 ms  ✓
Batida Maracuja              alcohol    0.77    379.9 ms  ✓
Tomi Losos/Pastrva           food       0.94    410.8 ms  ✗ (label: pets)
Violeta XXL Ručnici          household  0.57    368.8 ms  ✗ (label: toiletries)


## Claude Haiku 5.5 (chat completions through OpenRouter)

A chat LLM has no question type for a choice. We put the category list in the system prompt, ask
for JSON with a fixed schema and parse the answer. The confidence is a number that the model
writes in its answer, not a probability for each category.

To use Qwen3.8 27B instead, set `LLM = "qwen"`.

In [10]:
LLMS = {
    # The provider is pinned, so the requests go to the same backend as in the full run.
    "haiku": {"model": "anthropic/claude-haiku-5.5-20261007",
              "provider": {"order": ["anthropic"], "allow_fallbacks": False, "require_parameters": True}},
    "qwen": {"model": "qwen/qwen3.8-27b-20260814",
             "provider": {"order": ["near-ai"], "quantizations": ["fp8"], "allow_fallbacks": False,
                          "require_parameters": True}},
}
LLM = "haiku"

category_lines = "\n".join(f"- {c}: {d}" for c, d in DESCRIPTIONS.items())
SYSTEM = f"""{INSTRUCTIONS} The user message gives the product name and its quantity with the unit.

Categories:
{category_lines}

Answer with the category id and your confidence: the probability from 0 to 1 that the category is correct."""

SCHEMA = {
    "type": "object",
    "properties": {
        "category": {"type": "string", "enum": list(DESCRIPTIONS)},
        "confidence": {"type": "number", "description": "Probability from 0 to 1 that the category is correct"},
    },
    "required": ["category", "confidence"],
    "additionalProperties": False,
}

openrouter = httpx.Client(base_url="https://openrouter.ai/api/v1", timeout=60,
                          headers={"Authorization": f"Bearer {os.environ['OPENROUTER_API_KEY']}"})


def llm_answer(p):
    r = openrouter.post("/chat/completions", json={
        **LLMS[LLM],
        "messages": [
            {"role": "system", "content": SYSTEM},
            {"role": "user", "content": f"Product: {p['name']}\nQuantity: {p['quantity']} {p['unit']}"},
        ],
        "response_format": {"type": "json_schema",
                            "json_schema": {"name": "classification", "strict": True, "schema": SCHEMA}},
        "reasoning": {"effort": "none"},  # no thinking step (Qwen thinks by default)
        "max_tokens": 200,
    })
    r.raise_for_status()
    return json.loads(r.json()["choices"][0]["message"]["content"])


llm_answer(products[0])

{'category': 'alcohol', 'confidence': 0.97}

In [11]:
def ask_llm(p):
    a = llm_answer(p)
    return a["category"], a["confidence"]


live[LLM] = classify(ask_llm)

Pelinkovac                   alcohol    0.97    892.8 ms  ✓
Kraš Express Kakao Napitak   drinks     0.93   1447.9 ms  ✓
Batida Maracuja              alcohol    0.90   1303.2 ms  ✓
Tomi Losos/Pastrva           food       0.95   1458.0 ms  ✗ (label: pets)
Violeta XXL Ručnici          household  0.80   1045.8 ms  ✗ (label: toiletries)


## Same answers as the full run?

The full run stored every answer in `results/predictions.csv.gz`. Jev and Luna gave the same
answers in our reruns. Haiku and Qwen can differ in a few answers.

In [12]:
stored = pd.read_csv("results/predictions.csv.gz", dtype={"ean": str})
stored = stored[(stored.variant == "main") & stored.ean.isin(DEMO)].pivot(index="ean", columns="method",
                                                                           values="label")
check = demo[["name"]].copy()
for method, answers in live.items():
    check[method] = answers
    check[f"{method}, full run"] = stored.loc[demo.index, method]
check

,name,jev,"jev, full run",luna,"luna, full run",haiku,"haiku, full run"
ean,,,,,,,
3850158406356,Pelinkovac,alcohol,alcohol,alcohol,alcohol,alcohol,alcohol
3850102126903,Kraš Express Kakao Napitak,drinks,drinks,drinks,drinks,drinks,drinks
4003310017177,Batida Maracuja,alcohol,alcohol,alcohol,alcohol,alcohol,alcohol
4003024157053,Tomi Losos/Pastrva,food,food,food,food,food,food
3859892288129,Violeta XXL Ručnici,household,household,household,household,household,household


## TF-IDF + logistic regression

The classic baseline, trained on the 20,777 labelled products in the train split:

- TF-IDF turns each name into a vector with one number per n-gram (a piece of 2 to 5
  characters), weighted so that n-grams that occur in many names count less.
- Logistic regression learns a weight for each n-gram and each category.

The settings are the same as in `tfidf.py`, so the test score is the same as in the full run
(accuracy 96.7%, macro-F1 0.916).

In [14]:
%%time
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline


def text(p):
    """Input of the trained models: name and pack size. For one product or a whole table."""
    return p["name"] + " " + p["quantity"] + " " + p["unit"]


train, test = df[df.split == "train"], df[df.split == "test"]
tfidf = make_pipeline(
    TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), sublinear_tf=True, min_df=2),
    LogisticRegression(C=10, max_iter=3000),
)
tfidf.fit(text(train), train.category)

CPU times: user 5min 55s, sys: 561 ms, total: 5min 55s
Wall time: 19.2 s


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('tfidfvectorizer', ...), ('logisticregression', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[object](10,)","['alcohol','baby','cleaning',...,'household','pets','toiletries']"
,"analyzer analyzer: {'word', 'char', 'char_wb'} or callable, default='word'Whether the feature should be made of word or character n-grams.Option 'char_wb' creates character n-grams only from text insideword boundaries; n-grams at the edges of words are padded with space.If a callable is passed it is used to extract the sequence of featuresout of the raw, unprocessed input... versionchanged:: 0.21 Since v0.21, if ``input`` is ``'filename'`` or ``'file'``, the data is first read from the file and then passed to the given callable analyzer.",'char_wb'
,"ngram_range ngram_range: tuple (min_n, max_n), default=(1, 1)The lower and upper boundary of the range of n-values for differentn-grams to be extracted. All values of n such that min_n <= n <= max_nwill be used. For example an ``ngram_range`` of ``(1, 1)`` means onlyunigrams, ``(1, 2)`` means unigrams and bigrams, and ``(2, 2)`` meansonly bigrams.Only applies if ``analyzer`` is not callable.","(2, ...)"
,"min_df min_df: float or int, default=1When building the vocabulary ignore terms that have a documentfrequency strictly lower than the given threshold. This value is alsocalled cut-off in the literature.If float in range of [0.0, 1.0], the parameter represents a proportionof documents, integer absolute counts.This parameter is ignored if vocabulary is not None.",2
,"sublinear_tf sublinear_tf: bool, default=FalseApply sublinear tf scaling, i.e. replace tf with 1 + log(tf).",True
,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If `'file'`, the sequence items must have a 'read' method (file-like object) that is called to fetch the bytes in memory.- If `'content'`, the input is expected to be a sequence of items that can be of type s

In [15]:
from sklearn.metrics import accuracy_score, f1_score


def score(pred):
    print(f"test: {len(test):,} products, accuracy {accuracy_score(test.category, pred):.1%}, "
          f"macro-F1 {f1_score(test.category, pred, average='macro'):.3f}")


score(tfidf.predict(text(test)))

test: 5,938 products, accuracy 96.7%, macro-F1 0.916


In [16]:
def ask_tfidf(p):
    probs = tfidf.predict_proba([text(p)])[0]
    return tfidf.classes_[probs.argmax()], probs.max()


live["tfidf"] = classify(ask_tfidf)

Pelinkovac                   alcohol    0.99      0.7 ms  ✓
Kraš Express Kakao Napitak   food       0.88      0.6 ms  ✗ (label: drinks)
Batida Maracuja              drinks     0.44      0.5 ms  ✗ (label: alcohol)
Tomi Losos/Pastrva           pets       0.69      0.4 ms  ✓
Violeta XXL Ručnici          toiletries 0.83      0.4 ms  ✓


## BERTić, fine-tuned

BERTić is a BERT-type language model, pre-trained on Bosnian, Croatian, Montenegrin and Serbian
text. Fine-tuning puts a new classification layer on top and trains the whole model on our
labelled products.

- One epoch (one pass over the 20,777 training products) takes about a minute on a GPU.
- The full run trains 10 epochs and keeps the epoch with the best macro-F1 on the dev split
  (accuracy 96.5%, macro-F1 0.913). After one epoch, accuracy is lower and macro-F1 is much
  lower: the model has not learned the small categories yet.
- The trained model stays in memory. The training work directory is `out/notebook/`, and
  `just clean` deletes it.

The load report is expected. MISSING: the weights of the new classification layer, which
training sets. UNEXPECTED: the weights of the pre-training head, which classification does not
use.

In [17]:
from datasets import Dataset
from transformers import AutoModelForSequenceClassification, AutoTokenizer, set_seed

BASE_MODEL = "classla/bcms-bertic"
REVISION = "5db9755d6152ec6403c0201223e4848bd1b98a48"  # the same model files as in the full run
labels = list(categories.id)
label2id = {c: i for i, c in enumerate(labels)}

set_seed(42)  # the new classification layer starts from random weights
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, revision=REVISION)
model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL, revision=REVISION, num_labels=len(labels), id2label=dict(enumerate(labels)), label2id=label2id,
)
train_ds = Dataset.from_dict({"text": list(text(train)), "label": [label2id[c] for c in train.category]})
train_ds = train_ds.map(lambda b: tokenizer(b["text"], max_length=64, truncation=True), batched=True)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

ElectraForSequenceClassification LOAD REPORT from: classla/bcms-bertic
Key                                               | Status     | 
--------------------------------------------------+------------+-
discriminator_predictions.dense_prediction.weight | UNEXPECTED | 
electra.embeddings.position_ids                   | UNEXPECTED | 
discriminator_predictions.dense.weight            | UNEXPECTED | 
discriminator_predictions.dense.bias              | UNEXPECTED | 
discriminator_predictions.dense_prediction.bias   | UNEXPECTED | 
classifier.dense.bias                             | MISSING    | 
classifier.out_proj.weight                        | MISSING    | 
classifier.out_proj.bias                          | MISSING    | 
classifier.dense.weight                           | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint

Map:   0%|          | 0/20777 [00:00<?, ? examples/s]

In [18]:
%%time
from transformers import DataCollatorWithPadding, Trainer, TrainingArguments

args = TrainingArguments(
    output_dir="out/notebook", num_train_epochs=1, per_device_train_batch_size=32,
    learning_rate=2e-5, warmup_steps=0.1, weight_decay=0.01, save_strategy="no",
    logging_steps=100, report_to="none", seed=42,
)
trainer = Trainer(model=model, args=args, train_dataset=train_ds, data_collator=DataCollatorWithPadding(tokenizer))
trainer.train()

Step,Training Loss
100,1.862673
200,1.041707
300,0.765236
400,0.599527
500,0.547979
600,0.512862


CPU times: user 54.2 s, sys: 352 ms, total: 54.5 s
Wall time: 54.4 s


TrainOutput(global_step=650, training_loss=0.8541392810528095, metrics={'train_runtime': 54.0459, 'train_samples_per_second': 384.432, 'train_steps_per_second': 12.027, 'total_flos': 233317124911080.0, 'train_loss': 0.8541392810528095, 'epoch': 1.0})

In [19]:
from transformers import pipeline

bertic = pipeline("text-classification", model=model, tokenizer=tokenizer, device=model.device)
score([r["label"] for r in bertic(list(text(test)), batch_size=256)])

test: 5,938 products, accuracy 88.1%, macro-F1 0.558


In [20]:
def ask_bertic(p):
    r = bertic(text(p))[0]
    return r["label"], r["score"]


live["bertic"] = classify(ask_bertic)

Pelinkovac                   alcohol    0.84     49.5 ms  ✓
Kraš Express Kakao Napitak   food       0.96      3.5 ms  ✗ (label: drinks)
Batida Maracuja              drinks     0.53      3.0 ms  ✗ (label: alcohol)
Tomi Losos/Pastrva           food       0.41      3.0 ms  ✗ (label: pets)
Violeta XXL Ručnici          household  0.39      3.0 ms  ✗ (label: toiletries)


## All answers

In [21]:
demo[["name", "category"]].assign(**live)

,name,category,jev,luna,haiku,tfidf,bertic
ean,,,,,,,
3850158406356,Pelinkovac,alcohol,alcohol,alcohol,alcohol,alcohol,alcohol
3850102126903,Kraš Express Kakao Napitak,drinks,drinks,drinks,drinks,food,food
4003310017177,Batida Maracuja,alcohol,alcohol,alcohol,alcohol,drinks,drinks
4003024157053,Tomi Losos/Pastrva,pets,food,food,food,pets,food
3859892288129,Violeta XXL Ručnici,toiletries,household,household,household,toiletries,household
